# Post-Shock Holdings Adjustment

This final test asks whether mutual funds actively rotate toward or away from firms with known rate sensitivity after FOMC surprises. The surprise occurs during quarter *q*, holdings move during *q*, and the quarter-end filing becomes observable in *q+1*. These holdings are outcomes—not same-quarter predictors.

## 1. Setup and frozen design

For each fund, the passive counterfactual drifts its pre-shock weights using realized stock returns. The response is actual ending weights minus passive weights, projected onto firm rate betas estimated strictly before the quarter. Positive values mean active trading tilted toward firms with more positive historical reactions to tightening.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name=='economic_shock_sensitivity': ROOT=ROOT.parent
EXT=ROOT/'economic_shock_sensitivity'; sys.path.insert(0,str(EXT/'src'))
from post_shock_holdings import construct_response, hac_quarterly_regression
RESULTS=EXT/'results'/'post_shock_holdings'; RESULTS.mkdir(parents=True,exist_ok=True)
RETURNS=pd.read_parquet(ROOT/'active_passive_extension/measures/firm_returns_v2_total_return/quarterly_firm_returns_2005_2024.parquet')
ACTIVENESS=pd.read_parquet(ROOT/'active_passive_extension/measures/s12_activeness_master_80q.parquet')
SHOCKS=pd.read_parquet(EXT/'data/processed/fomc_event_returns/fomc_surprises_2005_2024.parquet')
SHOCKS['period']=SHOCKS.event_date.dt.to_period('Q')
quarter_shocks=SHOCKS.groupby('period',as_index=False).agg(shock_stmt=('shock_stmt','sum'),shock_me=('shock_me','sum'),events=('event_date','size'))
PERIODS=pd.period_range('2010Q1','2024Q4',freq='Q')

## 2. Construct return-adjusted responses

Quarter checkpoints make this restart-safe. This is the only relatively heavy cell; it reads two holdings snapshots per quarter but processes one quarter at a time.

In [ ]:
parts=[]
for number,period in enumerate(PERIODS,1):
    path=RESULTS/f'{str(period).lower()}_fund_response.parquet'
    if path.exists(): frame=pd.read_parquet(path)
    else:
        frame=construct_response(ROOT,period,RETURNS,ACTIVENESS); frame.to_parquet(path,index=False)
    parts.append(frame); print(f'[{number:02d}/{len(PERIODS)}] {period}: {len(frame):,} funds')
responses=pd.concat(parts,ignore_index=True)
responses['period']=pd.PeriodIndex(responses.quarter,freq='Q')
responses=responses.merge(quarter_shocks,on='period',how='left',validate='many_to_one')
responses.to_parquet(RESULTS/'fund_quarter_responses.parquet',index=False)
display(responses[['active_rate_tilt_change','return_adjusted_turnover_recomputed','beta_firms','pre_coverage','post_coverage','weight_balance']].describe())

## 3. Quarterly response tests

Funds are averaged within quarter so the effective sample is the number of shock quarters, not the number of funds. Newey–West standard errors allow serial dependence. STMT is primary; ME is a shock-measure robustness check.

In [ ]:
quarterly=responses.groupby(['period','shock_stmt','shock_me'],as_index=False).agg(mean_active_rate_tilt_change=('active_rate_tilt_change','mean'),median_active_rate_tilt_change=('active_rate_tilt_change','median'),funds=('investor_key','nunique'))
rows=[]
for sample,mask in [('full',quarterly.period<=pd.Period('2024Q4')),('final_2021_2024',quarterly.period>=pd.Period('2021Q1'))]:
    for shock in ['shock_stmt','shock_me']:
        rows.append({'sample':sample,'shock':shock,**hac_quarterly_regression(quarterly.loc[mask],'mean_active_rate_tilt_change',shock)})
tests=pd.DataFrame(rows); quarterly.to_parquet(RESULTS/'quarterly_mean_responses.parquet',index=False); tests.to_parquet(RESULTS/'quarterly_response_tests.parquet',index=False)
display(tests)

## 4. Behavioral heterogeneity

These comparisons are secondary. They describe whether investor types adjust differently; the central embedding result remains the ex-ante firm-level prediction test.

In [ ]:
by_style=responses.dropna(subset=['behavioral_group']).groupby(['period','shock_stmt','behavioral_group'],as_index=False).agg(mean_active_rate_tilt_change=('active_rate_tilt_change','mean'),funds=('investor_key','nunique'))
style_rows=[]
for style,frame in by_style.groupby('behavioral_group'):
    if frame.period.nunique()>=20: style_rows.append({'behavioral_group':style,**hac_quarterly_regression(frame,'mean_active_rate_tilt_change','shock_stmt')})
style_tests=pd.DataFrame(style_rows).sort_values('slope',ascending=False); by_style.to_parquet(RESULTS/'behavioral_quarterly_responses.parquet',index=False); style_tests.to_parquet(RESULTS/'behavioral_response_tests.parquet',index=False)
display(style_tests)

## 5. Integrity gates and interpretation boundary

A significant coefficient documents post-shock rebalancing consistent with rate sensitivity. It does not turn disclosed post-quarter holdings into an ex-ante forecast.

In [ ]:
gates=pd.DataFrame([{'gate':'60 quarters constructed','passed':responses.period.nunique()==60},{'gate':'weight changes balance','passed':responses.weight_balance.abs().quantile(.99)<1e-8},{'gate':'median pre-universe coverage >= 75%','passed':responses.pre_coverage.median()>=.75},{'gate':'median post-universe coverage >= 75%','passed':responses.post_coverage.median()>=.75},{'gate':'quarterly tests finite','passed':np.isfinite(tests[['slope','standard_error','t_statistic']].to_numpy(float)).all()}])
gates.to_parquet(RESULTS/'integrity_gates.parquet',index=False); display(gates); assert gates.passed.all()
print('Post-shock holdings-response extension complete.')